# SCEX-CPDP: Complete Google Colab Research Notebook

**Research title:** An Explainable and Cost-Sensitive Machine Learning Framework for Cross-Project Software Defect Prediction  
**Framework:** Stability-aware Cost-sensitive Explainable Cross-Project Defect Prediction (SCEX-CPDP)

This notebook is a reproducible starter implementation. It downloads the Jureczko/PROMISE corpus, inspects and prepares the data, evaluates baselines using Leave-One-Project-Out (LOPO), tunes a cost-sensitive model threshold using source-project-only grouped out-of-fold predictions, computes effort-aware recall using LOC as a proxy, and assesses SHAP explanation stability.

**Research integrity:** No results are pre-filled or fabricated. Run the notebook and report only observed outputs. The false-negative/false-positive cost ratio is an illustrative assumption, not measured industry cost. LOC is an effort proxy, not actual testing time or money. SHAP explains model associations, not causation.

## 1. Runtime setup
Run in Google Colab. CPU is sufficient for these classical ML models.

In [ ]:
!pip -q install liac-arff imbalanced-learn shap scipy scikit-learn pandas numpy matplotlib seaborn

## 2. Imports and reproducibility

In [ ]:
import os, re, json, random, warnings, subprocess, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, balanced_accuracy_score, average_precision_score,
    roc_auc_score, confusion_matrix)
from scipy.stats import spearmanr, wilcoxon
import shap

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")
print("Python", platform.python_version(), "| pandas", pd.__version__,
      "| sklearn", __import__("sklearn").__version__, "| SHAP", shap.__version__)

## 3. Download Jureczko/PROMISE backup corpus

In [ ]:
REPO_DIR = Path("/content/PROMISE-backup")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/feiwww/PROMISE-backup.git", str(REPO_DIR)], check=True)

data_files = []
for ext in ("*.csv", "*.arff", "*.CSV", "*.ARFF"):
    data_files.extend(REPO_DIR.rglob(ext))
print("Candidate files:", len(data_files))
for p in sorted(data_files)[:60]:
    print(p.relative_to(REPO_DIR))

## 4. Load CSV and ARFF files

Project inference uses known project tokens in filenames/paths. Inspect the mapping in later cells and correct `infer_project()` if necessary. A wrong project mapping invalidates LOPO evaluation.

In [ ]:
try:
    import arff
except ImportError:
    import liac_arff as arff

KNOWN_PROJECTS = ["ant","camel","ivy","jedit","log4j","lucene",
                  "poi","synapse","velocity","xalan","xerces"]

def infer_project(path):
    s = str(path).lower()
    for p in sorted(KNOWN_PROJECTS, key=len, reverse=True):
        if re.search(rf"(^|[^a-z0-9]){re.escape(p)}([^a-z0-9]|$)", s):
            return p
    stem = Path(path).stem.lower()
    return re.split(r"[_\-.\s]+", stem)[0] or "unknown"

def load_one_file(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, low_memory=False)
    if path.suffix.lower() == ".arff":
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            obj = arff.load(f)
        return pd.DataFrame(obj["data"], columns=[a[0] for a in obj["attributes"]])
    return None

frames, load_errors = [], []
for path in sorted(data_files):
    try:
        frame = load_one_file(path)
        if frame is not None and not frame.empty:
            frame["__source_file"] = str(path.relative_to(REPO_DIR))
            frame["__project"] = infer_project(path)
            frames.append(frame)
    except Exception as e:
        load_errors.append((str(path), repr(e)))

print("Loaded:", len(frames), "| errors:", len(load_errors))
for item in load_errors[:10]: print(item)
if not frames:
    raise RuntimeError("No data files loaded. Check repository layout and file formats.")
raw = pd.concat(frames, ignore_index=True, sort=False)
print("Raw shape:", raw.shape)
display(raw.head())

## 5. Inspect projects, columns, and target

In [ ]:
display(raw.groupby("__project").size().sort_values(ascending=False).to_frame("rows"))
print("Columns:")
for col in raw.columns: print(repr(col))
print("Possible target columns:",
      [c for c in raw.columns if c.lower() in {"bug","bugs","defect","defects","number_of_bugs"}])
display(raw.groupby("__project")["__source_file"].nunique().to_frame("source_files"))

## 6. Clean data and construct binary defect target

Default target is `bug` (defect count); label 1 means `bug > 0`, label 0 means no recorded defects. The notebook excludes identifiers and metadata, retaining numeric metric columns only. Missing-value imputation is performed inside each training pipeline.

In [ ]:
TARGET_COL = next((c for c in raw.columns if c.lower() == "bug"), None)
if TARGET_COL is None:
    TARGET_COL = next((c for c in raw.columns if c.lower() in
                       {"bugs","defect","defects","number_of_bugs"}), None)
if TARGET_COL is None:
    raise ValueError("Defect target not found. Inspect raw.columns and set TARGET_COL manually.")

target_num = pd.to_numeric(raw[TARGET_COL], errors="coerce")
valid = target_num.notna()
data = raw.loc[valid].copy().reset_index(drop=True)
target_num = target_num.loc[valid].reset_index(drop=True)
data["__label"] = (target_num > 0).astype(int)

excluded = {TARGET_COL.lower(), "__label", "__project", "__source_file",
            "name","classname","class","class_name","file","filename",
            "version","project","package","id"}
feature_cols = []
for c in data.columns:
    if c.lower() in excluded or c.startswith("__"): continue
    numeric = pd.to_numeric(data[c], errors="coerce")
    if numeric.notna().any():
        feature_cols.append(c)
        data[c] = numeric
feature_cols = [c for c in feature_cols if data[c].notna().any()]
if not feature_cols: raise ValueError("No numeric metric features detected.")

X = data[feature_cols].replace([np.inf, -np.inf], np.nan)
y = data["__label"].astype(int)
groups = data["__project"].astype(str)
keep = X.notna().any(axis=1)
X, y, groups, data = X.loc[keep], y.loc[keep], groups.loc[keep], data.loc[keep]
X, y, groups, data = (z.reset_index(drop=True) for z in (X,y,groups,data))

print("Target:", TARGET_COL, "| features:", len(feature_cols), "| X:", X.shape)
print("Projects:", sorted(groups.unique()))
display(y.value_counts().rename(index={0:"non-defective",1:"defective"}).to_frame("rows"))
print("Overall defect prevalence:", round(y.mean(),4))

## 7. Validate project mapping and feature missingness

In [ ]:
for project, files in data.groupby("__project")["__source_file"].unique().items():
    print("\n", project)
    for f in list(files)[:15]: print("  ", f)
display(X.isna().mean().sort_values(ascending=False).to_frame("missing_fraction").head(30))
display(pd.DataFrame({"rows":groups.value_counts(),
    "defect_rate":y.groupby(groups).mean(),
    "class_count":y.groupby(groups).nunique()}).sort_index())

## 8. Define baseline pipelines and evaluation metrics

In [ ]:
def make_models(seed=SEED):
    return {
      "Dummy": Pipeline([("imputer",SimpleImputer(strategy="median")),
                         ("model",DummyClassifier(strategy="prior"))]),
      "LogisticRegression": Pipeline([("imputer",SimpleImputer(strategy="median")),
          ("scale",StandardScaler()),
          ("model",LogisticRegression(max_iter=3000,class_weight="balanced",random_state=seed))]),
      "RandomForest": Pipeline([("imputer",SimpleImputer(strategy="median")),
          ("model",RandomForestClassifier(n_estimators=400,min_samples_leaf=2,
              class_weight="balanced",random_state=seed,n_jobs=-1))]),
      "ExtraTrees": Pipeline([("imputer",SimpleImputer(strategy="median")),
          ("model",ExtraTreesClassifier(n_estimators=400,min_samples_leaf=2,
              class_weight="balanced",random_state=seed,n_jobs=-1))]),
      "RBF_SVM": Pipeline([("imputer",SimpleImputer(strategy="median")),
          ("scale",StandardScaler()),
          ("model",SVC(kernel="rbf",probability=True,class_weight="balanced",random_state=seed))])
    }

def safe_auc(y_true, score, kind="roc"):
    if len(np.unique(y_true)) < 2: return np.nan
    try:
        return roc_auc_score(y_true,score) if kind=="roc" else average_precision_score(y_true,score)
    except Exception: return np.nan

def recall_at_effort(y_true, score, effort, fraction=0.20):
    yt, sc, ef = np.asarray(y_true), np.asarray(score), np.asarray(effort,dtype=float)
    ef = np.where(np.isfinite(ef) & (ef>0), ef, 1.0)
    if yt.sum()==0 or ef.sum()<=0: return np.nan
    order = np.argsort(-sc); budget = fraction*ef.sum()
    used, found = 0.0, 0.0
    for i in order:
        if used >= budget: break
        used += ef[i]; found += yt[i]
    return found/yt.sum()

def metric_dict(y_true, pred, score, loc=None):
    m = {"Accuracy":accuracy_score(y_true,pred),
         "Precision":precision_score(y_true,pred,zero_division=0),
         "Recall":recall_score(y_true,pred,zero_division=0),
         "F1":f1_score(y_true,pred,zero_division=0),
         "MCC":matthews_corrcoef(y_true,pred),
         "Balanced_Accuracy":balanced_accuracy_score(y_true,pred),
         "PR_AUC":safe_auc(y_true,score,"pr"),
         "ROC_AUC":safe_auc(y_true,score,"roc")}
    if loc is not None: m["Recall_at_20pct_LOC"] = recall_at_effort(y_true,score,loc,0.20)
    return m

## 9. Cost-sensitive model and source-only threshold tuning

Default illustrative cost ratio: `C_FN=5`, `C_FP=1`. Class weights and threshold are configurable. Threshold tuning uses grouped out-of-fold predictions on source projects only; the held-out project is never used to choose the threshold.

In [ ]:
C_FN, C_FP = 5.0, 1.0
COST_CLASS_WEIGHT = {0:1.0, 1:C_FN/C_FP}
THRESHOLD_GRID = np.linspace(0.05,0.95,91)

def make_cost_model(seed=SEED):
    return Pipeline([("imputer",SimpleImputer(strategy="median")),
      ("model",RandomForestClassifier(n_estimators=500,min_samples_leaf=2,
          class_weight=COST_CLASS_WEIGHT,random_state=seed,n_jobs=-1))])

def total_cost(y_true,pred):
    tn,fp,fn,tp = confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    return C_FN*fn + C_FP*fp

def tune_threshold_source_only(Xs,ys,source_groups,seed=SEED):
    unique = np.unique(source_groups)
    if len(unique)<2 or len(np.unique(ys))<2: return 0.5, None
    cv = GroupKFold(n_splits=min(5,len(unique)))
    try:
        oof = cross_val_predict(make_cost_model(seed),Xs,ys,groups=source_groups,
                                cv=cv,method="predict_proba")[:,1]
    except Exception as e:
        print("Threshold tuning fallback (0.5):",repr(e))
        return 0.5,None
    costs=[total_cost(ys,(oof>=t).astype(int)) for t in THRESHOLD_GRID]
    return float(THRESHOLD_GRID[int(np.argmin(costs))]),oof

## 10. Leave-One-Project-Out (LOPO) evaluation

All releases/files mapped to a project are held out together. This avoids training on one release and testing on another release of the same project. This is cross-project evaluation, not within-project chronological prediction.

In [ ]:
loc_col = next((c for c in X.columns if c.lower() in {"loc","locode","loc_code","linesofcode"}),None)
print("LOC effort proxy:",loc_col)
results, pred_frames, fold_models = [], [], {}
project_names = sorted(groups.unique())
if len(project_names)<2: raise ValueError("LOPO needs at least two correctly identified projects.")

for heldout in project_names:
    test_mask = (groups==heldout).to_numpy()
    train_mask = ~test_mask
    Xtr,ytr = X.loc[train_mask],y.loc[train_mask]
    Xte,yte = X.loc[test_mask],y.loc[test_mask]
    source_groups = groups.loc[train_mask].to_numpy()
    if len(np.unique(ytr))<2:
        print("Skipping",heldout,"because source training data has one class.")
        continue
    loc_test = (Xte[loc_col].fillna(Xte[loc_col].median()).to_numpy()
                if loc_col else None)

    for name,model in make_models().items():
        model.fit(Xtr,ytr)
        score=model.predict_proba(Xte)[:,1]
        pred=(score>=0.5).astype(int)
        results.append({"heldout_project":heldout,"model":name,"threshold":0.5,
            "n_train":len(Xtr),"n_test":len(Xte),"defect_rate_test":float(yte.mean()),
            **metric_dict(yte,pred,score,loc_test)})
        pred_frames.append(pd.DataFrame({"heldout_project":heldout,"model":name,
            "y_true":yte.to_numpy(),"score":score,"prediction":pred}))

    threshold,oof=tune_threshold_source_only(Xtr,ytr,source_groups)
    cmodel=make_cost_model()
    cmodel.fit(Xtr,ytr)
    cscore=cmodel.predict_proba(Xte)[:,1]
    cpred=(cscore>=threshold).astype(int)
    results.append({"heldout_project":heldout,"model":"SCEX_CPDP_CostSensitiveRF",
        "threshold":threshold,"n_train":len(Xtr),"n_test":len(Xte),
        "defect_rate_test":float(yte.mean()),"Expected_Cost":total_cost(yte,cpred),
        **metric_dict(yte,cpred,cscore,loc_test)})
    pred_frames.append(pd.DataFrame({"heldout_project":heldout,
        "model":"SCEX_CPDP_CostSensitiveRF","y_true":yte.to_numpy(),
        "score":cscore,"prediction":cpred}))
    fold_models[heldout]=cmodel

results_df=pd.DataFrame(results)
predictions_df=pd.concat(pred_frames,ignore_index=True) if pred_frames else pd.DataFrame()
display(results_df)
print("LOPO projects completed:",results_df.heldout_project.nunique() if len(results_df) else 0)

## 11. Aggregate performance and visualize

In [ ]:
if results_df.empty: raise RuntimeError("No folds completed; inspect target/project mapping.")
metrics=[c for c in ["Accuracy","Precision","Recall","F1","MCC","Balanced_Accuracy",
                     "PR_AUC","ROC_AUC","Recall_at_20pct_LOC"] if c in results_df]
summary_mean=results_df.groupby("model")[metrics].mean(numeric_only=True)
summary_std=results_df.groupby("model")[metrics].std(numeric_only=True)
display(summary_mean.round(4))
display(summary_std.round(4))
ax=summary_mean[[c for c in ["F1","MCC","PR_AUC","Recall"] if c in summary_mean]].plot(
    kind="bar",figsize=(12,5))
ax.set_title("LOPO performance: mean across held-out projects")
ax.set_ylabel("Score")
plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()

## 12. SHAP explanations for the cost-sensitive Random Forest

SHAP is an explanation method, not a novel model contribution by itself. It describes model behaviour and associations; it does not establish causal effects. To control runtime, the notebook samples at most 300 test instances per project.

In [ ]:
MAX_SHAP_ROWS=300
shap_importance_by_project={}
shap_records=[]

for heldout,model in fold_models.items():
    idx=np.where((groups==heldout).to_numpy())[0]
    Xfold=X.iloc[idx]
    Xsample=Xfold.sample(min(len(Xfold),MAX_SHAP_ROWS),random_state=SEED)
    transformed=model.named_steps["imputer"].transform(Xsample)
    rf=model.named_steps["model"]
    explainer=shap.TreeExplainer(rf)
    values=explainer.shap_values(transformed)
    if isinstance(values,list):
        positive=values[1] if len(values)>1 else values[0]
    else:
        arr=np.asarray(values)
        positive=arr[:,:,1] if arr.ndim==3 and arr.shape[2]>1 else (arr[:,:,0] if arr.ndim==3 else arr)
    importance=pd.Series(np.abs(positive).mean(axis=0),index=X.columns).sort_values(ascending=False)
    shap_importance_by_project[heldout]=importance
    for feature,val in importance.items():
        shap_records.append({"heldout_project":heldout,"feature":feature,"mean_abs_shap":val})

shap_importance_df=pd.DataFrame(shap_records)
if not shap_importance_df.empty:
    global_shap=shap_importance_df.groupby("feature").mean_abs_shap.mean().sort_values(ascending=False)
    display(global_shap.head(20).to_frame("mean_abs_SHAP"))
    global_shap.head(20).sort_values().plot(kind="barh",figsize=(9,6),
        title="Mean absolute SHAP importance across LOPO folds")
    plt.tight_layout(); plt.show()
else:
    print("No SHAP results available.")

## 13. Explanation stability across held-out projects

In [ ]:
pairs=[]
projects=list(shap_importance_by_project)
for i in range(len(projects)):
    for j in range(i+1,len(projects)):
        p1,p2=projects[i],projects[j]
        a,b=shap_importance_by_project[p1],shap_importance_by_project[p2]
        common=sorted(set(a.index)&set(b.index))
        if len(common)>=3:
            rho,pval=spearmanr(a.loc[common],b.loc[common])
            pairs.append({"project_1":p1,"project_2":p2,
                          "spearman_rho":rho,"p_value":pval,
                          "common_features":len(common)})
stability_df=pd.DataFrame(pairs)
display(stability_df)
if not stability_df.empty:
    print("Mean pairwise explanation rank correlation:",stability_df.spearman_rho.mean())

## 14. Explanation stability across random seeds (one source split)

In [ ]:
SEEDS_FOR_STABILITY=[11,22,33,44,55]
seed_importances={}
eligible=list(fold_models)
if eligible:
    chosen=eligible[0]
    testmask=(groups==chosen).to_numpy()
    Xsource,ysource=X.loc[~testmask],y.loc[~testmask]
    for seed in SEEDS_FOR_STABILITY:
        m=make_cost_model(seed); m.fit(Xsource,ysource)
        seed_importances[seed]=pd.Series(m.named_steps["model"].feature_importances_,index=X.columns)
    seed_corr=pd.DataFrame(seed_importances).corr(method="spearman")
    display(seed_corr)
    print("Mean off-diagonal seed rank correlation:",
        seed_corr.where(~np.eye(len(seed_corr),dtype=bool)).stack().mean())
else:
    print("No eligible fold.")

## 15. Paired statistical comparison

This example compares project-level metric values using Wilcoxon signed-rank. The paired unit is the held-out project. With few projects, interpret p-values cautiously; for multiple models use an appropriate omnibus test and post-hoc correction (e.g., Friedman plus Holm-adjusted comparisons). Consider reporting effect sizes such as Cliff's delta.

In [ ]:
def paired_wilcoxon(table,model_a,model_b,metric="MCC"):
    pivot=table.pivot(index="heldout_project",columns="model",values=metric)
    if model_a not in pivot or model_b not in pivot:
        return {"error":"Model missing from result table."}
    paired=pivot[[model_a,model_b]].dropna()
    if len(paired)<3: return {"error":f"Only {len(paired)} paired projects."}
    diff=paired[model_a]-paired[model_b]
    if np.allclose(diff,0):
        return {"n_projects":len(paired),"statistic":0.0,"p_value":1.0}
    stat,p=wilcoxon(paired[model_a],paired[model_b])
    return {"n_projects":len(paired),"statistic":stat,"p_value":p,
            "median_difference_a_minus_b":float(diff.median())}

print(paired_wilcoxon(results_df,"SCEX_CPDP_CostSensitiveRF","RandomForest","MCC"))

## 16. Export tables and run metadata

In [ ]:
OUT_DIR=Path("/content/SCEX_CPDP_results")
OUT_DIR.mkdir(exist_ok=True)
results_df.to_csv(OUT_DIR/"lopo_metrics_by_project.csv",index=False)
summary_mean.to_csv(OUT_DIR/"metrics_mean_by_model.csv")
summary_std.to_csv(OUT_DIR/"metrics_std_by_model.csv")
predictions_df.to_csv(OUT_DIR/"lopo_predictions.csv",index=False)
if not shap_importance_df.empty:
    shap_importance_df.to_csv(OUT_DIR/"shap_importance_by_project.csv",index=False)
if not stability_df.empty:
    stability_df.to_csv(OUT_DIR/"shap_project_stability.csv",index=False)

metadata={"framework":"SCEX-CPDP","seed":SEED,"target_column":TARGET_COL,
 "features":list(X.columns),"projects":sorted(groups.unique().tolist()),
 "C_FN":C_FN,"C_FP":C_FP,"class_weight":COST_CLASS_WEIGHT,
 "rows":int(len(X)),"n_features":int(X.shape[1]),
 "python":platform.python_version(),"pandas":pd.__version__,
 "sklearn":__import__("sklearn").__version__,"shap":shap.__version__}
with open(OUT_DIR/"run_metadata.json","w") as f: json.dump(metadata,f,indent=2)
print("Saved:",[p.name for p in OUT_DIR.iterdir()])

## 17. Download result bundle (Google Colab)

In [ ]:
import shutil
from google.colab import files
zip_file=shutil.make_archive("/content/SCEX_CPDP_results","zip",OUT_DIR)
files.download(zip_file)

## 18. Optional separate benchmark: NASA MDP

NASA MDP datasets are a different dataset family with a different metric schema. **Do not concatenate NASA MDP records with Jureczko records.** Treat NASA as a separate benchmark, harmonize its target/features independently, and report separate results.

Repository: https://github.com/ApoorvaKrisna/NASA-promise-dataset-repository

In [ ]:
NASA_DIR=Path("/content/nasa_promise")
if not NASA_DIR.exists():
    subprocess.run(["git","clone","--depth","1",
      "https://github.com/ApoorvaKrisna/NASA-promise-dataset-repository.git",
      str(NASA_DIR)],check=True)
nasa_files=list(NASA_DIR.rglob("*.csv"))+list(NASA_DIR.rglob("*.arff"))
print("NASA candidate files:",len(nasa_files))
for p in nasa_files[:60]: print(p.relative_to(NASA_DIR))

## 19. Manuscript reporting checklist

Before writing the Results section, record:
- Repository URL, retrieval date/commit, exact downloaded files and dataset versions.
- File-to-project mapping and any exclusions; confirm all releases of a project stay in one LOPO fold.
- Exact feature columns, target conversion (`bug > 0`), missingness and preprocessing.
- Project-wise sample sizes and defect prevalence.
- Models, hyperparameters, class weights, threshold tuning procedure and cost assumptions.
- Per-project results, mean and standard deviation; do not report only pooled row-level metrics.
- Definition of effort-aware recall and the LOC proxy.
- SHAP version, explainer, sampling cap and explanation-stability method.
- Statistical tests, paired unit, multiple-comparison correction and effect sizes.

**Contribution framing:** The defensible methodological combination is project-level leakage-resistant validation, source-only cost-sensitive threshold selection, effort-aware prioritization, and explanation-stability assessment. SHAP alone is not novel. Avoid causal claims and do not claim performance gains until experiments and statistical tests support them.

## 20. Troubleshooting and limitations

- **No target column:** inspect `raw.columns` and set `TARGET_COL` manually.
- **Only one project detected:** fix `infer_project()` before evaluation.
- **Wrong project mapping:** correct it; otherwise LOPO may leak data.
- **Slow runtime:** temporarily reduce tree estimators and SHAP sample size for debugging; restore before final runs.
- **One-class test fold:** ROC-AUC/PR-AUC can be undefined and are recorded as NaN.
- **SHAP API issue:** inspect SHAP version and output array shape; update SHAP if needed.
- **Dataset comparability:** metric definitions/values may differ across releases and tools.
- **External validity:** these are older public defect datasets; conclusions may not generalize to modern software teams.

Do not submit notebook outputs as research findings until the notebook has been executed, mappings audited, and results independently checked.